# Skills + MCP Agent Loop

This notebook builds and runs, end-to-end, a **minimal 4-stage agent loop** that
ties together `01-agent-skills`'s `Skill`/`SkillRegistry` (skill *selection*, i.e.
which set of instructions is relevant to a task) and `02-model-context-protocol`'s
JSON-RPC MCP server/client (tool *discovery and invocation*, i.e. actually doing
something in the world).

Neither piece alone is a working agent:

- Topic 1 can tell you "this task is about arithmetic" but has no notion of a
  *tool* to call, or arguments to call it with.
- Topic 2 can discover and invoke `add(a, b)` but has no notion of *which* task
  a client should even be asking for `add` versus `word_count` versus something
  else — that decision lived entirely outside the MCP client in Topic 2's
  notebook (a human picked which tool to call, by hand, for each cell).

This notebook is the first place in the curriculum where a **task string** (not
a human) drives which skill is selected, which tool that skill points toward,
what arguments get extracted from the string, and what real result comes back —
a real, if toy-scale, closed loop.

**Binding constraint carried over from Topics 1 and 2:** no live LLM/external
service call anywhere. Skill selection reuses Topic 1's deterministic
keyword-overlap heuristic; the MCP server is the same in-repo subprocess as
Topic 2; argument extraction is plain regex, explicitly labeled as a stand-in
for what an LLM would normally do (parse a natural-language task into a
structured tool call).


## Stage 0 setup: reuse Topic 1's `Skill`/`SkillRegistry` verbatim

This is a direct copy of the `Skill` dataclass and `SkillRegistry` class from
`01-agent-skills/001_progressive_disclosure_skill_selection.ipynb` (cells 3 and
7) — not a reimplementation. Same fields, same `select()` keyword-overlap
heuristic, same `STOPWORDS`/`normalize_tokens` helpers.

In [1]:
import json
import random
import re
import subprocess
import sys
from dataclasses import dataclass, field
from pathlib import Path


In [2]:
SKILL_SOURCE_TEMPLATE = """---
name: {name}
description: {description}
---

{body}"""


@dataclass
class Skill:
    name: str
    description: str
    body: str

    @property
    def full_context_chars(self) -> int:
        """Characters consumed if this skill's FULL instructions are loaded."""
        return len(self.name) + len(self.description) + len(self.body)

    @property
    def summary_context_chars(self) -> int:
        """Characters consumed by just the always-visible name + one-line description."""
        return len(self.name) + len(self.description)

    @classmethod
    def from_markdown(cls, text: str) -> "Skill":
        """Minimal frontmatter parser: pulls `name:` / `description:` out of the
        `---` block, treats everything after the closing `---` as the body."""
        match = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.DOTALL)
        if not match:
            raise ValueError("Not a valid skill file: missing frontmatter block")
        frontmatter, body = match.group(1), match.group(2).strip()
        fields = {}
        for line in frontmatter.splitlines():
            key, _, value = line.partition(":")
            fields[key.strip()] = value.strip()
        return cls(name=fields["name"], description=fields["description"], body=body)

    def to_markdown(self) -> str:
        return SKILL_SOURCE_TEMPLATE.format(
            name=self.name, description=self.description, body=self.body
        )


STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "for", "in", "on", "with", "is", "are",
    "this", "that", "it", "its", "as", "by", "from", "into", "not", "be", "than",
}


def normalize_tokens(text: str) -> set[str]:
    words = re.findall(r"[a-z0-9]+", text.lower())
    return {w for w in words if w not in STOPWORDS and len(w) > 2}


@dataclass
class SkillRegistry:
    skills: dict[str, Skill] = field(default_factory=dict)

    def register(self, skill: Skill) -> None:
        self.skills[skill.name] = skill

    def all_descriptions(self) -> dict[str, str]:
        """The always-visible tier: name -> one-line description for every skill."""
        return {name: s.description for name, s in self.skills.items()}

    def select(self, task: str) -> tuple[str, dict[str, int]]:
        """Deterministic, non-LLM keyword-overlap selection.
        Returns (best_skill_name, {skill_name: overlap_score}) for inspection."""
        task_tokens = normalize_tokens(task)
        scores = {}
        for name, skill in self.skills.items():
            desc_tokens = normalize_tokens(skill.description)
            scores[name] = len(task_tokens & desc_tokens)
        best = max(scores, key=lambda n: scores[n])
        return best, scores

    def load_full_body(self, name: str) -> str:
        """The second tier: only called AFTER a skill is selected."""
        return self.skills[name].body


print("Skill / SkillRegistry reused verbatim from 01-agent-skills.")


Skill / SkillRegistry reused verbatim from 01-agent-skills.


## Toy skills for this topic

Two skills are the point of this notebook — `arithmetic-calculation` and
`text-analysis` — each written so its description vocabulary overlaps with the
kind of task it should route to, and each with a body that (honestly) tells the
agent to reach for an external tool rather than compute the answer itself. Three
more skills are carried over unchanged from Topic 1 (`csv-cleaning`,
`email-drafting`, `code-review-checklist`) purely as **distractors** — they make
skill selection a real choice among five options instead of a trivial pick
between two.

In [3]:
arithmetic_skill = Skill(
    name="arithmetic-calculation",
    description="Perform arithmetic calculations - add, sum, subtract numbers given in a task",
    body=(
        "# Arithmetic Calculation\n\n"
        "1. Identify the numbers involved in the task and the operation requested.\n"
        "2. Do NOT compute the result yourself -- an external calculation tool exists "
        "for this and is more reliable than ad-hoc reasoning.\n"
        "3. Call the `add` tool with the extracted numbers as arguments `a` and `b`.\n"
        "4. Report the tool's returned value as the final answer, unchanged.\n"
    ),
)

text_analysis_skill = Skill(
    name="text-analysis",
    description="Analyze text - count words, reverse strings, inspect a passage of text",
    body=(
        "# Text Analysis\n\n"
        "1. Identify what property of the text is being asked for (a word count, a "
        "reversed string, etc.) and the text itself.\n"
        "2. Call the matching external tool (`word_count` or `reverse_string`) with the "
        "extracted text as the `text` argument.\n"
        "3. Report the tool's returned value as the final answer, unchanged.\n"
    ),
)

csv_cleaning_skill = Skill(
    name="csv-cleaning",
    description="Clean messy CSV data - fix encodings, missing values, inconsistent column types, duplicate rows",
    body="# CSV Cleaning\n\n(body omitted for this topic -- distractor skill, no matching MCP tool)",
)

email_drafting_skill = Skill(
    name="email-drafting",
    description="Draft professional emails - replies, follow-ups, meeting requests, status updates",
    body="# Email Drafting\n\n(body omitted for this topic -- distractor skill, no matching MCP tool)",
)

code_review_skill = Skill(
    name="code-review-checklist",
    description="Review a pull request for correctness bugs, missing tests, and style issues",
    body="# Code Review Checklist\n\n(body omitted for this topic -- distractor skill, no matching MCP tool)",
)

registry = SkillRegistry()
for s in (arithmetic_skill, text_analysis_skill, csv_cleaning_skill, email_drafting_skill, code_review_skill):
    registry.register(s)

print("Registered skills (always-visible tier):")
for name, desc in registry.all_descriptions().items():
    print(f"  {name}: {desc}")


Registered skills (always-visible tier):
  arithmetic-calculation: Perform arithmetic calculations - add, sum, subtract numbers given in a task
  text-analysis: Analyze text - count words, reverse strings, inspect a passage of text
  csv-cleaning: Clean messy CSV data - fix encodings, missing values, inconsistent column types, duplicate rows
  email-drafting: Draft professional emails - replies, follow-ups, meeting requests, status updates
  code-review-checklist: Review a pull request for correctness bugs, missing tests, and style issues


## The glue: skill -> candidate tool names

Nothing in Topic 1 or Topic 2 connects a *skill* to a *tool name* — that
connection has to be authored somewhere, and this map is exactly that: a small,
explicit, human-authored table saying "if this skill is selected, these are the
tool names worth even considering." This is the piece that was **missing**
before this topic (see notes.md's "Why simpler approaches fail") — Topic 1's
registry and Topic 2's server never talked to each other.

Two distractor skills (`csv-cleaning`, `email-drafting`) map to an empty tool
list (no MCP tool matches them). `code-review-checklist` deliberately maps to a
tool name, `"lint_code"`, that does **not** exist on this MCP server — this is
used later in "Failure modes" to demonstrate a real integration mismatch caught
by the discovery step.

In [4]:
SKILL_TO_TOOLS = {
    "arithmetic-calculation": ["add"],
    "text-analysis": ["word_count", "reverse_string"],
    "csv-cleaning": [],
    "email-drafting": [],
    "code-review-checklist": ["lint_code"],  # does not exist on this server -- see Failure modes
}


## Stage 2: discover tools from the MCP server (reused verbatim from Topic 2)

Same subprocess-over-stdio client as `02-model-context-protocol`'s notebook:
`mcp_server.py` (a local copy in this topic's folder, for self-containedness) is
spawned as a real subprocess, and `list_tools` is a real JSON-RPC 2.0 request
over its stdin/stdout pipes.

In [5]:
SERVER_PATH = Path.cwd() / "mcp_server.py"
assert SERVER_PATH.exists(), f"expected server script at {SERVER_PATH}"

proc = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1,  # line-buffered
)
print(f"Spawned real subprocess: pid={proc.pid}, cmd={[sys.executable, str(SERVER_PATH)]}")

_next_id = 0

def send_request(method, params=None, print_transcript=True):
    """Send one real JSON-RPC 2.0 request to the subprocess server and return
    the parsed JSON-RPC response dict. Prints the raw wire transcript."""
    global _next_id
    _next_id += 1
    request = {"jsonrpc": "2.0", "id": _next_id, "method": method, "params": params or {}}
    request_line = json.dumps(request)

    proc.stdin.write(request_line + "\n")
    proc.stdin.flush()
    response_line = proc.stdout.readline().strip()
    response = json.loads(response_line)

    if print_transcript:
        print("--> SENT     :", request_line)
        print("<-- RECEIVED :", response_line)
        print()
    return response


Spawned real subprocess: pid=424004, cmd=['/home/yashwanth-aravind/ml-course/python-bootcamp/.venv/bin/python', '/home/yashwanth-aravind/ml-course/python-bootcamp/15-agent-skills-and-mcp/03-skills-and-mcp-agent-loop/mcp_server.py']


In [6]:
list_tools_response = send_request("list_tools")
discovered_tools = {t["name"]: t for t in list_tools_response["result"]["tools"]}
print(f"Server exposes {len(discovered_tools)} real tools:")
for name, spec in discovered_tools.items():
    print(f"  - {name}: {spec['description']}")


--> SENT     : {"jsonrpc": "2.0", "id": 1, "method": "list_tools", "params": {}}
<-- RECEIVED : {"jsonrpc": "2.0", "id": 1, "result": {"tools": [{"name": "add", "description": "Add two numbers together and return their sum.", "inputSchema": {"type": "object", "properties": {"a": {"type": "number", "description": "First addend"}, "b": {"type": "number", "description": "Second addend"}}, "required": ["a", "b"]}}, {"name": "word_count", "description": "Count the number of whitespace-separated words in a string.", "inputSchema": {"type": "object", "properties": {"text": {"type": "string", "description": "The text to count words in"}}, "required": ["text"]}}, {"name": "reverse_string", "description": "Reverse the characters of a string.", "inputSchema": {"type": "object", "properties": {"text": {"type": "string", "description": "The text to reverse"}}, "required": ["text"]}}]}}

Server exposes 3 real tools:
  - add: Add two numbers together and return their sum.
  - word_count: Count the nu

## Stage 3: deterministic argument extraction

Simple regex parsing over the raw task string — an explicit, honest stand-in
for what an LLM would normally do (turn a natural-language task into a
structured tool call). No language model is involved anywhere in this
notebook.

In [7]:
NUMBER_RE = re.compile(r"-?\d+(?:\.\d+)?")


def extract_add_args(task: str) -> dict:
    """Pull the first two numbers out of the task string, in order, as a and b."""
    numbers = NUMBER_RE.findall(task)
    if len(numbers) < 2:
        raise ValueError(
            f"extract_add_args: expected at least two numbers in the task, found {numbers!r} in {task!r}"
        )
    a, b = numbers[0], numbers[1]
    return {"a": float(a) if "." in a else int(a), "b": float(b) if "." in b else int(b)}


COUNT_WORDS_RE = re.compile(r"count.*?words?.*?in:?\s*(.+)$", re.IGNORECASE)
REVERSE_RE = re.compile(r"reverse.*?:?\s*(.+)$", re.IGNORECASE)


def extract_word_count_args(task: str) -> dict:
    match = COUNT_WORDS_RE.search(task)
    if not match:
        raise ValueError(f"extract_word_count_args: pattern not found in {task!r}")
    return {"text": match.group(1).strip()}


def extract_reverse_args(task: str) -> dict:
    match = REVERSE_RE.search(task)
    if not match:
        raise ValueError(f"extract_reverse_args: pattern not found in {task!r}")
    return {"text": match.group(1).strip()}


EXTRACTORS = {
    "add": extract_add_args,
    "word_count": extract_word_count_args,
    "reverse_string": extract_reverse_args,
}
print("Extractors registered for:", list(EXTRACTORS.keys()))


Extractors registered for: ['add', 'word_count', 'reverse_string']


## Stage 4 + the full loop

Given a task string, run all four stages in order and return a record of what
happened at each stage:

1. **Select** a skill from `registry` (Topic 1).
2. **Narrow** candidate tools to `SKILL_TO_TOOLS[selected_skill]`, then pick the
   single best-matching tool name (by the *same* keyword-overlap scoring
   Topic 1 uses for skills) from among the tool names that are BOTH in that
   narrowed candidate list AND actually present in `discovered_tools` (Topic 2's
   `list_tools` result) -- this second check is exactly what catches the
   `code-review-checklist` -> `lint_code` mismatch in "Failure modes" below.
3. **Extract** real arguments from the task string via the matching regex
   extractor.
4. **Invoke** the tool over the live MCP subprocess and return its real result.

In [8]:
def select_tool_from_candidates(task: str, candidate_names: list[str], tool_catalog: dict) -> tuple[str | None, dict]:
    """Same keyword-overlap scoring Topic 1 uses for skills, applied to tool
    descriptions instead, restricted to `candidate_names`."""
    task_tokens = normalize_tokens(task)
    scores = {}
    for name in candidate_names:
        if name not in tool_catalog:
            continue  # candidate tool not actually available -- excluded, not guessed at
        desc_tokens = normalize_tokens(tool_catalog[name]["description"])
        scores[name] = len(task_tokens & desc_tokens)
    if not scores:
        return None, scores
    best = max(scores, key=lambda n: scores[n])
    return best, scores


def agent_loop(task: str, verbose: bool = True) -> dict:
    """The 4-stage skills+MCP agent loop, run once for a single toy task."""
    record = {"task": task}

    # Stage 1: skill selection (Topic 1's SkillRegistry, reused verbatim)
    selected_skill, skill_scores = registry.select(task)
    record["selected_skill"] = selected_skill
    record["skill_scores"] = skill_scores

    # Stage 2: narrow to this skill's candidate tools, then pick one from what
    # was actually discovered via list_tools (Topic 2's MCP client, reused verbatim)
    candidates = SKILL_TO_TOOLS.get(selected_skill, [])
    selected_tool, tool_scores = select_tool_from_candidates(task, candidates, discovered_tools)
    record["candidate_tools"] = candidates
    record["selected_tool"] = selected_tool
    record["tool_scores"] = tool_scores

    if verbose:
        print(f"TASK: {task!r}")
        print(f"  [stage 1] selected skill : {selected_skill}  (scores: {skill_scores})")
        print(f"  [stage 2] candidate tools: {candidates}  -> selected tool: {selected_tool}")

    if selected_tool is None:
        record["error"] = f"no available tool for skill {selected_skill!r} among candidates {candidates!r}"
        if verbose:
            print(f"  [ABORT] {record['error']}")
        return record

    # Stage 3: deterministic argument extraction
    try:
        arguments = EXTRACTORS[selected_tool](task)
    except ValueError as exc:
        record["error"] = str(exc)
        if verbose:
            print(f"  [stage 3] argument extraction FAILED: {exc}")
        return record
    record["arguments"] = arguments
    if verbose:
        print(f"  [stage 3] extracted arguments: {arguments}")

    # Stage 4: real invocation over the live MCP subprocess
    response = send_request("call_tool", {"name": selected_tool, "arguments": arguments}, print_transcript=False)
    if "error" in response:
        record["error"] = response["error"]
        if verbose:
            print(f"  [stage 4] tool call FAILED: {response['error']}")
        return record
    record["result"] = response["result"]["value"]
    if verbose:
        print(f"  [stage 4] tool result: {record['result']}")
        print()
    return record


## Running the loop on toy tasks

Three toy tasks, run end-to-end with real output at every stage.

In [9]:
toy_tasks = [
    "please add 17 and 25 for me",
    "count the words in: the quick brown fox",
    "reverse this text: model context protocol",
]

results = [agent_loop(t) for t in toy_tasks]


TASK: 'please add 17 and 25 for me'
  [stage 1] selected skill : arithmetic-calculation  (scores: {'arithmetic-calculation': 1, 'text-analysis': 0, 'csv-cleaning': 0, 'email-drafting': 0, 'code-review-checklist': 0})
  [stage 2] candidate tools: ['add']  -> selected tool: add
  [stage 3] extracted arguments: {'a': 17, 'b': 25}
  [stage 4] tool result: 42

TASK: 'count the words in: the quick brown fox'
  [stage 1] selected skill : text-analysis  (scores: {'arithmetic-calculation': 0, 'text-analysis': 2, 'csv-cleaning': 0, 'email-drafting': 0, 'code-review-checklist': 0})
  [stage 2] candidate tools: ['word_count', 'reverse_string']  -> selected tool: word_count
  [stage 3] extracted arguments: {'text': 'the quick brown fox'}
  [stage 4] tool result: 4

TASK: 'reverse this text: model context protocol'
  [stage 1] selected skill : text-analysis  (scores: {'arithmetic-calculation': 0, 'text-analysis': 2, 'csv-cleaning': 0, 'email-drafting': 0, 'code-review-checklist': 0})
  [stage 2] can

In [10]:
for r in results:
    assert "result" in r, f"expected a real result for task {r['task']!r}, got {r}"
print("All three toy tasks completed with real results:")
for r in results:
    print(f"  {r['task']!r} -> skill={r['selected_skill']!r}, tool={r['selected_tool']!r}, args={r['arguments']}, result={r['result']!r}")


All three toy tasks completed with real results:
  'please add 17 and 25 for me' -> skill='arithmetic-calculation', tool='add', args={'a': 17, 'b': 25}, result=42
  'count the words in: the quick brown fox' -> skill='text-analysis', tool='word_count', args={'text': 'the quick brown fox'}, result=4
  'reverse this text: model context protocol' -> skill='text-analysis', tool='reverse_string', args={'text': 'this text: model context protocol'}, result='locotorp txetnoc ledom :txet siht'


## Connecting this loop to `14-multi-agent-systems/02-orchestration-patterns`

That topic's notes.md formalizes manager/worker orchestration as four explicit
steps: **task decomposition** ($T \to T_1, \ldots, T_n$), **delegation** (each
$T_i$ sent to exactly one worker), **worker execution** ($R_i = f(T_i)$), and
**result aggregation** ($R = \text{aggregate}(R_1, \ldots, R_n)$).

`agent_loop` above is **the degenerate $n=1$ case of exactly that same
pattern**, run by a single agent against itself instead of across a pool of
worker processes:

- **Decomposition ($T \to T_1$):** trivial here -- one task string is treated
  as exactly one subtask ($n=1$), so `combine` is the identity function. A
  richer agent would decompose "add 17 and 25, then count the words in the
  result" into two subtasks first; this notebook stops at $n=1$ deliberately,
  to isolate the skill-to-tool wiring from decomposition itself.
- **Delegation:** instead of an `Orchestrator` sending `T_i` to a `Worker`
  object over a message queue, "delegation" here is stage 1 + stage 2 of
  `agent_loop` -- selecting a skill (which *specialist* should handle this) and
  narrowing to a tool (which *concrete capability* that specialist should
  invoke). The "worker" being delegated to is the MCP server subprocess itself.
- **Worker execution ($R_i = f(T_i)$):** stage 4, the real `call_tool` JSON-RPC
  request/response over the live subprocess -- $f$ is whatever Python function
  `TOOL_FUNCTIONS[selected_tool]` runs inside `mcp_server.py`.
- **Aggregation ($R = \text{aggregate}(R_1)$):** trivial identity again at
  $n=1$ -- `agent_loop`'s return value already *is* the final answer, the same
  way a manager/worker system with exactly one worker has nothing left to
  combine after collecting that worker's single result.

The reason this is worth naming explicitly: everything Topic 14's
"Failure modes" observed about manager/worker systems at $n>1$ (a bottlenecked
orchestrator, per-worker coordination overhead) has a direct $n=1$ analogue
here -- the single agent *is* the orchestrator, and every one of the tasks
below is processed strictly sequentially through the same four stages, one
task fully resolved (or failed) before the next begins.

## Experiment: does narrowing tools by selected skill reduce wrong-tool invocations?

**Hypothesis:** as the number of tools registered on an MCP server grows, a
tool-selection heuristic that first narrows candidates to the ones relevant to
the *selected skill* (via `SKILL_TO_TOOLS`) makes fewer wrong-tool-invocation
errors than one that keyword-matches against **every** registered tool's
description, because the "consider everything" variant has more and more
superficially-similar tool descriptions to be confused by as the registry
grows, while the skill-narrowed variant's candidate pool for any given task
stays fixed in size regardless of how many *other* tools exist.

**Setup.** The 3 real tools (`add`, `word_count`, `reverse_string`) are
extended with **synthetic** toy tools -- programmatically generated, not
hand-written -- built the same way Topic 1's notebook synthesizes extra toy
skills: each synthetic tool's description is assembled by randomly recombining
a word pool. The pool is deliberately drawn from **both** the real tools' own
descriptions **and** the experiment task battery's own vocabulary (defined
below) -- an honest stand-in for a realistic large tool marketplace, where many
unrelated third-party tools' descriptions happen to share generic words
("count", "text", "words", "add", "numbers") with common task phrasings
without being the actually-correct tool for any of them. A pool built only
from the real tools' descriptions turned out, when tried first, to cap every
synthetic tool's overlap score at a tie with the real tool's own score at best
(ties are broken toward the real tool by insertion order) -- producing a flat
0% wrong-tool rate for BOTH variants at every $N$, which would have made this
experiment uninformative. The task+description pool below is what actually
lets a synthetic tool occasionally out-score the correct real tool by chance,
which is the substantive effect being measured.

In [11]:
# Toy task battery with a known-correct tool for each task, used to measure
# wrong-tool-invocation rate. All tasks route to arithmetic-calculation or
# text-analysis via registry.select -- SKILL_TO_TOOLS is unchanged from above.
EXPERIMENT_TASKS = [
    ("add 3 and 4", "add"),
    ("please add 10 and 20 for me", "add"),
    ("what is the sum of 5 and 6", "add"),
    ("add together 100 and 1", "add"),
    ("add 42 and 8 please", "add"),
    ("count the words in: hello world", "word_count"),
    ("count the words in: a quick test of this sentence", "word_count"),
    ("reverse this text: hello", "reverse_string"),
    ("reverse: model context protocol", "reverse_string"),
    ("count the words in: another short passage here", "word_count"),
]
print(f"{len(EXPERIMENT_TASKS)} toy tasks in the experiment battery, each with a known-correct tool.")


def make_synthetic_tool(i: int, word_pool: list[str]) -> dict:
    """Synthesize one plausible toy tool description by recombining a word
    pool -- same technique Topic 1 uses to synthesize extra toy skills. NOT a
    real, callable tool; used only to grow the *candidate pool* a tool-
    selection heuristic has to search."""
    rng = random.Random(2000 + i)
    desc_len = rng.randint(8, 14)
    description = " ".join(rng.choice(word_pool) for _ in range(desc_len))
    return {"name": f"synthetic-tool-{i:03d}", "description": description}


real_tool_specs = [{"name": n, "description": s["description"]} for n, s in discovered_tools.items()]

# Pool = real tools' own description vocabulary + the task battery's own
# vocabulary (see markdown above for why the pool is built this way).
_pool_text = " ".join(spec["description"] for spec in real_tool_specs) + " " + " ".join(t for t, _ in EXPERIMENT_TASKS)
SYNTHETIC_WORD_POOL = _pool_text.split()

synthetic_tools = [make_synthetic_tool(i, SYNTHETIC_WORD_POOL) for i in range(27)]
print(f"Synthesized {len(synthetic_tools)} additional toy tool descriptions (synthetic-tool-000..026)")
print(f"from a {len(SYNTHETIC_WORD_POOL)}-word pool ({len(set(SYNTHETIC_WORD_POOL))} unique words).")
print("Example:", synthetic_tools[0])


10 toy tasks in the experiment battery, each with a known-correct tool.
Synthesized 27 additional toy tool descriptions (synthetic-tool-000..026)
from a 84-word pool (54 unique words).
Example: {'name': 'synthetic-tool-000', 'description': 'their words for add model short Reverse please 3 here the'}


In [12]:
def build_tool_catalog(n_total: int) -> dict:
    """First 3 slots are the real tools; remaining slots are synthetic."""
    pool = real_tool_specs + synthetic_tools
    assert n_total <= len(pool), "not enough synthesized tools for this N"
    return {spec["name"]: spec for spec in pool[:n_total]}


In [13]:
def consider_everything_select(task: str, tool_catalog: dict) -> str | None:
    """Variant A: keyword-match against EVERY registered tool's description,
    no skill-based narrowing at all."""
    best, _ = select_tool_from_candidates(task, list(tool_catalog.keys()), tool_catalog)
    return best


def skill_narrowed_select(task: str, tool_catalog: dict) -> str | None:
    """Variant B: select a skill first (Topic 1's registry), narrow candidates
    to SKILL_TO_TOOLS[selected_skill], THEN keyword-match only within that
    narrowed, much smaller candidate set."""
    selected_skill, _ = registry.select(task)
    candidates = SKILL_TO_TOOLS.get(selected_skill, [])
    best, _ = select_tool_from_candidates(task, candidates, tool_catalog)
    return best


def wrong_tool_rate(select_fn, tool_catalog: dict) -> float:
    wrong = 0
    for task, correct_tool in EXPERIMENT_TASKS:
        picked = select_fn(task, tool_catalog)
        if picked != correct_tool:
            wrong += 1
    return wrong / len(EXPERIMENT_TASKS)


registry_sizes = [5, 15, 30]
experiment_results = []
for n in registry_sizes:
    catalog = build_tool_catalog(n)
    rate_everything = wrong_tool_rate(consider_everything_select, catalog)
    rate_narrowed = wrong_tool_rate(skill_narrowed_select, catalog)
    experiment_results.append((n, rate_everything, rate_narrowed))

print(f"{'N tools':>8}  {'consider-everything wrong-tool rate':>36}  {'skill-narrowed wrong-tool rate':>32}")
for n, rate_e, rate_n in experiment_results:
    print(f"{n:>8}  {rate_e:>35.0%}  {rate_n:>31.0%}")


 N tools   consider-everything wrong-tool rate    skill-narrowed wrong-tool rate
       5                                  40%                               0%
      15                                  80%                               0%
      30                                  80%                               0%


**Interpretation.** The skill-narrowed variant's wrong-tool rate stays at
0% across all three registry sizes, because `SKILL_TO_TOOLS` restricts its
candidate pool to at most 2 real tool names regardless of how many synthetic
tools exist elsewhere in the catalog -- the synthetic tools are never even
considered, since nothing routes a skill to them. The consider-everything
variant's rate, measured directly above (not assumed), rises sharply between
$N=5$ and $N=15$ and then largely plateaus toward $N=30$ -- see notes.md's
Experiment section for the exact numbers from this run and a discussion of why
the effect saturates rather than climbing to 100%: once even one synthetic
tool in the catalog out-scores the correct real tool for a given task, adding
still more synthetic tools cannot make that task's outcome any *more* wrong
than it already is, and this experiment's fixed, seeded pool of 27 synthetic
tools was already generating that many-way overlap collisions well before
$N=30$. `random.Random(2000 + i)` is seeded, so re-running this notebook
reproduces the exact same numbers.

## Failure modes

Two concrete, reproduced failures -- not hypothetical ones.

### Failure 1: a skill points at a tool that does not exist on this server

`code-review-checklist` maps to `"lint_code"` in `SKILL_TO_TOOLS`, but this MCP
server was never asked to register a `lint_code` tool. This is a real
integration mismatch: the skill author's assumption about what tools are
available drifted from what the server actually exposes. The discovery step
(`list_tools`, already run above into `discovered_tools`) is exactly what
catches this **before** any invocation is attempted.

In [14]:
mismatch_task = "please review this pull request for bugs and missing tests"
selected_skill, scores = registry.select(mismatch_task)
candidates = SKILL_TO_TOOLS.get(selected_skill, [])
print(f"TASK: {mismatch_task!r}")
print(f"  selected skill: {selected_skill}  (scores: {scores})")
print(f"  candidate tools per SKILL_TO_TOOLS: {candidates}")

missing = [name for name in candidates if name not in discovered_tools]
print(f"  candidates NOT present in list_tools() result: {missing}")
assert missing == ["lint_code"], "expected the lint_code mismatch to be present"

record = agent_loop(mismatch_task)
print()
print("agent_loop's own outcome for this task:", record.get("error", record))
assert "error" in record, "expected agent_loop to abort rather than guess at a nonexistent tool"
print()
print("Caught BEFORE invocation: candidate tool names are checked against the real")
print("list_tools() result (discovered_tools) inside select_tool_from_candidates --")
print("lint_code is filtered out for not actually existing, selected_tool comes back")
print("None, and agent_loop aborts with an explicit error instead of guessing.")


TASK: 'please review this pull request for bugs and missing tests'
  selected skill: code-review-checklist  (scores: {'arithmetic-calculation': 0, 'text-analysis': 0, 'csv-cleaning': 1, 'email-drafting': 0, 'code-review-checklist': 6})
  candidate tools per SKILL_TO_TOOLS: ['lint_code']
  candidates NOT present in list_tools() result: ['lint_code']
TASK: 'please review this pull request for bugs and missing tests'
  [stage 1] selected skill : code-review-checklist  (scores: {'arithmetic-calculation': 0, 'text-analysis': 0, 'csv-cleaning': 1, 'email-drafting': 0, 'code-review-checklist': 6})
  [stage 2] candidate tools: ['lint_code']  -> selected tool: None
  [ABORT] no available tool for skill 'code-review-checklist' among candidates ['lint_code']

agent_loop's own outcome for this task: no available tool for skill 'code-review-checklist' among candidates ['lint_code']

Caught BEFORE invocation: candidate tool names are checked against the real
list_tools() result (discovered_tools) in

### Failure 2: the argument-extraction regex fails on an unexpected phrasing

`extract_add_args` looks for digit sequences in the task string. It has no
notion of numbers spelled out as words, so a perfectly reasonable arithmetic
task phrased without digits fails extraction -- even though skill selection and
tool selection both succeed correctly.

In [15]:
oddly_phrased_task = "please add seventeen and twenty five for me"
selected_skill, _ = registry.select(oddly_phrased_task)
candidates = SKILL_TO_TOOLS.get(selected_skill, [])
selected_tool, _ = select_tool_from_candidates(oddly_phrased_task, candidates, discovered_tools)
print(f"TASK: {oddly_phrased_task!r}")
print(f"  [stage 1] selected skill: {selected_skill}  (correct -- still arithmetic-calculation)")
print(f"  [stage 2] selected tool : {selected_tool}  (correct -- still add, it DOES exist)")

record = agent_loop(oddly_phrased_task)
print()
print("agent_loop's own outcome for this task:", record.get("error"))
assert "error" in record and "extract_add_args" in record["error"]
print()
print("This is a genuine parsing failure, not a hypothetical: skill selection and")
print("tool selection both got the right answer, but the deterministic regex")
print("extractor -- an explicit stand-in for what an LLM would normally do -- has no")
print("notion of numbers spelled out as words, so stage 3 fails cleanly with a")
print("caught ValueError rather than silently passing wrong arguments to call_tool.")


TASK: 'please add seventeen and twenty five for me'
  [stage 1] selected skill: arithmetic-calculation  (correct -- still arithmetic-calculation)
  [stage 2] selected tool : add  (correct -- still add, it DOES exist)
TASK: 'please add seventeen and twenty five for me'
  [stage 1] selected skill : arithmetic-calculation  (scores: {'arithmetic-calculation': 1, 'text-analysis': 0, 'csv-cleaning': 0, 'email-drafting': 0, 'code-review-checklist': 0})
  [stage 2] candidate tools: ['add']  -> selected tool: add
  [stage 3] argument extraction FAILED: extract_add_args: expected at least two numbers in the task, found [] in 'please add seventeen and twenty five for me'

agent_loop's own outcome for this task: extract_add_args: expected at least two numbers in the task, found [] in 'please add seventeen and twenty five for me'

This is a genuine parsing failure, not a hypothetical: skill selection and
tool selection both got the right answer, but the deterministic regex
extractor -- an explicit 

## Real-world usage

This 4-stage loop, scaled up along exactly two axes, is architecturally what
production "tool-using LLM agent" systems actually do:

1. **Replace the deterministic keyword heuristics (both `SkillRegistry.select`
   and `select_tool_from_candidates`) with a real LLM call.** An LLM reading a
   task string, a list of available skill descriptions, and a list of
   available tool schemas, and choosing among them, is a strict generalization
   of the keyword-overlap scoring used throughout this notebook -- same
   inputs, same discrete choice being made, just a far more capable selector.
2. **Replace the toy in-process/subprocess MCP server with real external MCP
   servers** -- GitHub, Slack, Postgres, Google Drive, Notion, and the rest,
   the same class of servers this very Claude Code session has deferred tools
   for (`mcp__claude_ai_Notion__*`, `mcp__claude_ai_Google_Calendar__*`,
   named again here as in Topic 2's notes.md, never called, per this section's
   no-live-external-call constraint).

This is the curriculum's own capstone connection for `15-agent-skills-and-mcp`:
Claude Code itself, Anthropic's Agent SDK, and "tool-using LLM agent" systems
generally are built on precisely this decompose-select-extract-invoke loop --
an LLM in place of `registry.select`/`select_tool_from_candidates`, real MCP
servers in place of `mcp_server.py`, and the same discovery-before-invocation
discipline this notebook's Failure Mode 1 demonstrated catching a real
integration mismatch before it reached a tool call.

## Cleanup

In [16]:
proc.stdin.close()
proc.wait(timeout=5)
stderr_output = proc.stderr.read()
print(f"Subprocess exited with code {proc.returncode}")
if stderr_output:
    print("stderr:", stderr_output)


Subprocess exited with code 0
